# REFINED Climate Classification - Solution 1
## Aggressive Sampling + Class-Balanced Loss + Strong Regularization

**Target: 80%+ Macro F1 and Accuracy**

### Key Fixes:
1. **MASSIVE oversampling** (10x minority class)
2. **Class-balanced focal loss** with very high gamma
3. **Strong regularization** to prevent overfitting
4. **Lower learning rate** for better convergence
5. **More epochs** with patience
6. **Ensemble of different thresholds**

In [1]:
!pip install -q transformers==4.45.0 scikit-learn openpyxl pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.9 MB/s eta 0:00:00


In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    precision_recall_curve, confusion_matrix
)

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

print('✓ Libraries loaded')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

✓ Libraries loaded
PyTorch: 2.8.0+cu126
CUDA: True
GPU: Tesla P100-PCIE-16GB


In [5]:
class CFG:
    # Paths
    train_path = '/kaggle/input/datasets/hrithikmajumdaroff/climate-text-dataset/Human labelled_DTU.xlsx'
    test_path = '/kaggle/input/datasets/hrithikmajumdaroff/climate-text-dataset/Master file_10k papers.xlsx'
    output_dir = '/kaggle/working/'
    
    # Model
    model_name = 'microsoft/deberta-v3-base'
    max_length = 512
    
    # Training - AGGRESSIVE SETTINGS
    n_folds = 5
    n_epochs = 15  # More epochs
    batch_size = 4  # Smaller batch for stability
    grad_accum_steps = 4  # Effective batch = 16
    lr = 8e-6  # MUCH lower LR
    weight_decay = 0.1  # Strong weight decay
    warmup_ratio = 0.2
    max_grad_norm = 0.5  # Strong clipping
    dropout = 0.3  # Heavy dropout
    
    # MASSIVE oversampling
    minority_multiplier = 10  # Oversample Accept 10x
    
    # Class-balanced focal loss
    focal_gamma = 4.0  # Very high gamma
    focal_alpha = 0.85  # Heavy weight on minority
    
    # Hardware
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fp16 = True
    num_workers = 2
    
    early_stopping_patience = 5
    seed = 42

print('✓ Aggressive configuration set')

✓ Aggressive configuration set


In [6]:
# Load data
train_df = pd.read_excel(CFG.train_path, skiprows=1)
train_df.columns = [
    'Coder name', 'Article ID', 'Paper_Author/s', 'Paper title',
    'Year of publication', 'DOI', 'URL', 'Abstracts',
    'Accept/Reject', 'If Accept, identify theme'
]

train_df = train_df[train_df['Accept/Reject'].isin(['Accept', 'Reject'])].copy()
train_df['text'] = train_df['Abstracts'].fillna('')
train_df = train_df[train_df['text'].str.len() > 50].reset_index(drop=True)
train_df['label'] = (train_df['Accept/Reject'] == 'Accept').astype(int)

test_df = pd.read_excel(CFG.test_path)
test_df['text'] = test_df['Abstract'].fillna('')
test_df = test_df[test_df['text'].str.len() > 50].reset_index(drop=True)

print(f'Training: {len(train_df)}, Test: {len(test_df)}')
print(f'\nClass distribution:')
print(train_df['label'].value_counts())
print(f'Ratio: {train_df["label"].value_counts()[0] / train_df["label"].value_counts()[1]:.2f}:1')

Training: 1719, Test: 10175

Class distribution:
label
0    1520
1     199
Name: count, dtype: int64
Ratio: 7.64:1


In [7]:
class ClimateDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [8]:
class ClassBalancedFocalLoss(nn.Module):
    """Class-balanced focal loss for severe imbalance"""
    def __init__(self, alpha=0.85, gamma=4.0, samples_per_class=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
        # Class-balanced weights
        if samples_per_class is not None:
            effective_num = 1.0 - np.power(0.9999, samples_per_class)
            self.cb_weights = (1.0 - 0.9999) / effective_num
            self.cb_weights = self.cb_weights / self.cb_weights.sum() * len(samples_per_class)
        else:
            self.cb_weights = None
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        
        # Alpha weighting
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        # Focal term
        focal_weight = (1 - pt) ** self.gamma
        
        # Combine
        loss = alpha_t * focal_weight * ce_loss
        
        # Apply class-balanced weights if available
        if self.cb_weights is not None:
            cb_w = torch.tensor([self.cb_weights[t] for t in targets.cpu().numpy()],
                              device=inputs.device)
            loss = loss * cb_w
        
        return loss.mean()

In [9]:
class ClimateClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.3):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({
            'hidden_dropout_prob': dropout,
            'attention_probs_dropout_prob': dropout,
        })
        
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        
        hidden_size = self.config.hidden_size
        
        # Simple but effective classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 2)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        pooled = outputs.last_hidden_state[:, 0]
        logits = self.classifier(pooled)
        
        return logits

In [10]:
def train_epoch(model, dataloader, optimizer, scheduler, criterion, device, scaler=None):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    optimizer.zero_grad()
    
    for step, batch in enumerate(tqdm(dataloader, desc='Training')):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                loss = loss / CFG.grad_accum_steps
            
            scaler.scale(loss).backward()
            
            if (step + 1) % CFG.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
        else:
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss = loss / CFG.grad_accum_steps
            loss.backward()
            
            if (step + 1) % CFG.grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
        
        total_loss += loss.item() * CFG.grad_accum_steps
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)
    
    return avg_loss, f1, acc

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(dataloader), np.array(all_preds), np.array(all_probs), np.array(all_labels)

def find_optimal_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5, f1_scores[best_idx]

In [11]:
# Training with aggressive oversampling
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

fold_scores = []
fold_thresholds = []
oof_probs = np.zeros(len(train_df))
oof_preds = np.zeros(len(train_df))
models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    print(f'\n{"="*80}')
    print(f'FOLD {fold + 1}/{CFG.n_folds}')
    print(f'{"="*80}')
    
    # MASSIVE oversampling
    fold_train = train_df.iloc[train_idx].copy()
    majority = fold_train[fold_train['label'] == 0]
    minority = fold_train[fold_train['label'] == 1]
    
    print(f'Original: Reject={len(majority)}, Accept={len(minority)}')
    
    # Oversample minority 10x
    minority_oversampled = pd.concat(
        [minority] * CFG.minority_multiplier,
        ignore_index=True
    )
    
    balanced_train = pd.concat(
        [majority, minority_oversampled],
        ignore_index=True
    ).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f'After 10x oversampling: Total={len(balanced_train)}')
    print(f'Reject={len(balanced_train[balanced_train["label"]==0])}, Accept={len(balanced_train[balanced_train["label"]==1])}')
    print(f'New ratio: {len(balanced_train[balanced_train["label"]==0]) / len(balanced_train[balanced_train["label"]==1]):.2f}:1\n')
    
    # Datasets
    train_dataset = ClimateDataset(
        balanced_train['text'].values,
        balanced_train['label'].values,
        tokenizer,
        CFG.max_length
    )
    
    val_dataset = ClimateDataset(
        train_df.iloc[val_idx]['text'].values,
        train_df.iloc[val_idx]['label'].values,
        tokenizer,
        CFG.max_length
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG.batch_size * 2,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=True
    )
    
    # Model
    model = ClimateClassifier(CFG.model_name, dropout=CFG.dropout).to(CFG.device)
    
    # Class-balanced focal loss
    samples_per_class = balanced_train['label'].value_counts().sort_index().values
    criterion = ClassBalancedFocalLoss(
        alpha=CFG.focal_alpha,
        gamma=CFG.focal_gamma,
        samples_per_class=samples_per_class
    )
    
    # Optimizer with low LR
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.lr,
        weight_decay=CFG.weight_decay
    )
    
    # Scheduler
    num_training_steps = len(train_loader) * CFG.n_epochs // CFG.grad_accum_steps
    num_warmup_steps = int(num_training_steps * CFG.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    scaler = torch.cuda.amp.GradScaler() if CFG.fp16 else None
    
    # Training
    best_f1 = 0
    patience = 0
    
    for epoch in range(CFG.n_epochs):
        print(f'\nEpoch {epoch + 1}/{CFG.n_epochs}')
        
        train_loss, train_f1, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, CFG.device, scaler
        )
        print(f'Train - Loss: {train_loss:.4f}, F1: {train_f1:.4f}, Acc: {train_acc:.4f}')
        
        val_loss, val_preds, val_probs, val_labels = validate(
            model, val_loader, criterion, CFG.device
        )
        
        threshold, _ = find_optimal_threshold(val_labels, val_probs)
        val_preds_opt = (val_probs >= threshold).astype(int)
        
        val_f1 = f1_score(val_labels, val_preds_opt, average='macro')
        val_acc = accuracy_score(val_labels, val_preds_opt)
        
        print(f'Val - Loss: {val_loss:.4f}, F1: {val_f1:.4f}, Acc: {val_acc:.4f}, Thresh: {threshold:.4f}')
        print(classification_report(val_labels, val_preds_opt, target_names=['Reject', 'Accept'], digits=4))
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_threshold = threshold
            patience = 0
            torch.save(model.state_dict(), f'{CFG.output_dir}/fold{fold}_best.pth')
            print(f'✓ Best F1: {best_f1:.4f}')
        else:
            patience += 1
            if patience >= CFG.early_stopping_patience:
                print(f'Early stopping at epoch {epoch + 1}')
                break
    
    # Load best and predict
    model.load_state_dict(torch.load(f'{CFG.output_dir}/fold{fold}_best.pth'))
    _, _, val_probs_final, _ = validate(model, val_loader, criterion, CFG.device)
    
    oof_probs[val_idx] = val_probs_final
    oof_preds[val_idx] = (val_probs_final >= best_threshold).astype(int)
    
    fold_scores.append(best_f1)
    fold_thresholds.append(best_threshold)
    models.append(model)
    
    print(f'\nFold {fold + 1} - Best F1: {best_f1:.4f}\n')
    
    del train_dataset, val_dataset, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*80}')
print('FINAL RESULTS')
print(f'{"="*80}')
print(f'CV F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'Fold scores: {[f"{s:.4f}" for s in fold_scores]}')

oof_f1 = f1_score(train_df['label'], oof_preds, average='macro')
oof_acc = accuracy_score(train_df['label'], oof_preds)
print(f'\nOOF F1: {oof_f1:.4f}')
print(f'OOF Acc: {oof_acc:.4f}')
print('\n' + classification_report(train_df['label'], oof_preds, target_names=['Reject', 'Accept']))

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


FOLD 1/5
Original: Reject=1216, Accept=159
After 10x oversampling: Total=2806
Reject=1216, Accept=1590
New ratio: 0.76:1



pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Train - Loss: 0.0021, F1: 0.9833, Acc: 0.9837


Validation:   0%|          | 0/43 [00:00<?, ?it/s]

Val - Loss: 0.0233, F1: 0.7276, Acc: 0.8746, Thresh: 0.7243
              precision    recall  f1-score   support

      Reject     0.9485    0.9079    0.9277       304
      Accept     0.4615    0.6154    0.5275        39

    accuracy                         0.8746       343
   macro avg     0.7050    0.7616    0.7276       343
weighted avg     0.8931    0.8746    0.8822       343

Early stopping at epoch 12


Validation:   0%|          | 0/43 [00:00<?, ?it/s]


Fold 5 - Best F1: 0.7453


FINAL RESULTS
CV F1: 0.6889 ± 0.0455
Fold scores: ['0.6243', '0.7303', '0.6529', '0.6917', '0.7453']

OOF F1: 0.6839
OOF Acc: 0.8400

              precision    recall  f1-score   support

      Reject       0.94      0.87      0.91      1520
      Accept       0.38      0.59      0.46       199

    accuracy                           0.84      1719
   macro avg       0.66      0.73      0.68      1719
weighted avg       0.88      0.84      0.85      1719



In [12]:
# Test predictions
test_dataset = ClimateDataset(
    test_df['text'].values,
    np.zeros(len(test_df)),
    tokenizer,
    CFG.max_length
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size * 2,
    shuffle=False,
    num_workers=CFG.num_workers
)

all_test_probs = []

for model in models:
    model.eval()
    test_probs = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Test Prediction'):
            input_ids = batch['input_ids'].to(CFG.device)
            attention_mask = batch['attention_mask'].to(CFG.device)
            
            logits = model(input_ids, attention_mask)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            test_probs.append(probs)
    
    all_test_probs.append(np.concatenate(test_probs))

avg_test_probs = np.mean(all_test_probs, axis=0)
avg_threshold = np.mean(fold_thresholds)
test_preds = (avg_test_probs >= avg_threshold).astype(int)

test_df['Prediction'] = ['Accept' if p == 1 else 'Reject' for p in test_preds]
test_df['Confidence'] = avg_test_probs

test_df[['ID_New', 'Article Title', 'Prediction', 'Confidence']].to_csv(
    f'{CFG.output_dir}/refined_solution1_predictions.csv',
    index=False
)

print(f'\n✓ Predictions saved')
print(f'Accept rate: {(test_preds == 1).sum() / len(test_preds) * 100:.2f}%')
print(f'\n{"="*80}')
print('TARGET ACHIEVED!' if oof_f1 >= 0.80 and oof_acc >= 0.80 else 'CLOSE! Try running again or tuning hyperparameters')
print(f'{"="*80}')

Test Prediction:   0%|          | 0/1272 [00:00<?, ?it/s]

Test Prediction:   0%|          | 0/1272 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c4e3125c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c4e3125c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Test Prediction:   0%|          | 0/1272 [00:00<?, ?it/s]

Test Prediction:   0%|          | 0/1272 [00:00<?, ?it/s]

Test Prediction:   0%|          | 0/1272 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c4e3125c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e1c4e3125c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16


✓ Predictions saved
Accept rate: 11.80%

CLOSE! Try running again or tuning hyperparameters
